In [ ]:

!pip install -q langchain langchain-text-splitters langchain-chroma langchain-huggingface pypdf sentence-transformers
!pip install -q transformers accelerate bitsandbytes




In [ ]:
!pip install -q langchain-community

In [ ]:
resume_filename = "my_resume.pdf"

In [ ]:

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
loader = PyPDFLoader("my_resume.pdf")
pages = loader.load()

print(f"Loaded {len(pages)} pages")

# Split into chunks

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
chunks = splitter.split_documents(pages)
print(f"Created {len(chunks)} chunks")

# Create embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

# Store in vector database
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)



In [ ]:

!pip install -q langchain-groq

from langchain_groq import ChatGroq
import os


os.environ["GROQ_API_KEY"] = "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"

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.1
)


In [ ]:
!pip install -q langchain-community

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Custom prompt template
template = """You are an assistant that answers questions about a resume.
Use only the following context to answer. If you don't know, say "I don't see that information in the resume."

Context: {context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

# Helper function to format retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# This connects: Retriever → Format → Prompt → LLM → Output
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
question = "What programming languages and technologies are mentioned?"

print(f"Question: {question}")
print("Searching resume...\n")
docs = retriever.invoke(question)
answer = rag_chain.invoke(question)

print("Answer:")
print(answer)

print("\n" + "="*50)
print("SOURCES USED:")

for doc in docs:
    doc.page_content = doc.page_content.replace("pekrishnasinghbirana93@gmail.com",
                                                 "krishnasinghbirana93@gmail.com")
    # Also clean other artifacts for display
    doc.page_content = doc.page_content.replace("/envel⌢", "")